In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
spark.sql("""
CREATE TABLE IF NOT EXISTS silver.orders (
    order_id STRING,
    customer_id STRING,
    product_id STRING,
    quantity INT,
    total_amount DOUBLE,
    transaction_date DATE,
    order_status STRING,
    last_updated TIMESTAMP
)
USING DELTA
""")

StatementMeta(, f74239f2-a958-46c9-af83-a42917760223, 3, Finished, Available, Finished, False)

DataFrame[]

In [2]:
last_processed_df = spark.sql("SELECT MAX(last_updated) as last_processed FROM silver.orders")
last_processed_timestamp = last_processed_df.collect()[0]['last_processed']

if last_processed_timestamp is None:
    last_processed_timestamp = "1900-01-01T00:00:00.000+00:00"

StatementMeta(, f74239f2-a958-46c9-af83-a42917760223, 4, Finished, Available, Finished, False)

In [3]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW bronze_incremental_orders AS
SELECT *
FROM enterpriseRetail.retailLakehouse.bronze.orders WHERE ingestion_timestamp > '{last_processed_timestamp}'
""")

StatementMeta(, f74239f2-a958-46c9-af83-a42917760223, 5, Finished, Available, Finished, False)

DataFrame[]

In [4]:
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW silver_incremental_orders AS
SELECT
    transaction_id as order_id,
    customer_id,
    product_id,
    CASE
        WHEN quantity < 0 THEN 0
        ELSE quantity
    END AS quantity,
    CASE
        WHEN total_amount < 0 THEN 0
        ELSE total_amount
    END AS total_amount,
    CAST(transaction_date AS DATE) AS transaction_date,
    CASE
        WHEN quantity = 0 AND total_amount = 0 THEN 'Cancelled'
        WHEN quantity > 0 AND total_amount > 0 THEN 'Completed'
        ELSE 'In Progress'
    END AS order_status,
    CURRENT_TIMESTAMP() AS last_updated
FROM bronze_incremental_orders
WHERE transaction_date IS NOT NULL 
  AND customer_id IS NOT NULL 
  AND product_id IS NOT NULL
""")

StatementMeta(, f74239f2-a958-46c9-af83-a42917760223, 6, Finished, Available, Finished, False)

DataFrame[]

In [5]:
spark.sql("""
MERGE INTO silver.orders target
USING silver_incremental_orders source
ON target.order_id = source.order_id
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *
""")

StatementMeta(, f74239f2-a958-46c9-af83-a42917760223, 7, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [6]:
spark.sql("SELECT * FROM silver.orders LIMIT 10").show()

StatementMeta(, f74239f2-a958-46c9-af83-a42917760223, 8, Finished, Available, Finished, False)

+---------+-----------+----------+--------+------------+----------------+------------+--------------------+
| order_id|customer_id|product_id|quantity|total_amount|transaction_date|order_status|        last_updated|
+---------+-----------+----------+--------+------------+----------------+------------+--------------------+
|TRX000063|        234|        67|       2|      550.83|      2021-09-12|   Completed|2026-03-05 00:02:...|
|TRX000115|         58|       475|       2|      299.56|      2022-07-31|   Completed|2026-03-05 00:02:...|
|TRX000126|         29|       609|       2|      706.21|      2021-12-02|   Completed|2026-03-05 00:02:...|
|TRX000144|        122|       202|       2|      446.44|      2022-09-24|   Completed|2026-03-05 00:02:...|
|TRX000311|        378|       719|       2|      945.18|      2020-02-19|   Completed|2026-03-05 00:02:...|
|TRX000326|        638|       602|       2|      687.12|      2021-07-10|   Completed|2026-03-05 00:02:...|
|TRX000544|        665|     